# API Processes to Charts

This notebook walks through an end-to-end REST API process using JSON data from Open-Meteo, then creates charts at the end.


## How to Use This Notebook

This notebook is designed to fetch hourly weather forecast data from the Open-Meteo API, process it, and visualise key metrics like temperature and humidity. It also demonstrates how to save the generated charts and raw data to files.

### Workflow:
1.  **Configuration**: Set up base URL, city coordinates, and other request parameters.
2.  **API Request**: Construct and execute an API request to Open-Meteo.
3.  **Data Processing**: Parse the JSON response into a Pandas DataFrame.
4.  **Data Summary**: Calculate basic statistics from the fetched data.
5.  **Visualisation**: Generate line plots for temperature and humidity over time, and a scatter plot for their relationship.
6.  **Output Saving**: Save the generated plots and the raw JSON response to local files.

### Customisation:
-   You can change the `city_key` variable in the "Initial Setup and Configuration" section to fetch data for a different city.
-   Adjust `forecast_days` to get forecasts for more or fewer days.
-   Modify plotting parameters to change chart appearance.

In [ ]:
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests

BASE_URL = 'https://api.open-meteo.com/v1/forecast'
CITY_COORDINATES = {
    'auckland': {'label': 'Auckland', 'latitude': -36.8485, 'longitude': 174.7633},
    'wellington': {'label': 'Wellington', 'latitude': -41.2866, 'longitude': 174.7756},
    'christchurch': {'label': 'Christchurch', 'latitude': -43.5321, 'longitude': 172.6362},
    'dunedin': {'label': 'Dunedin', 'latitude': -45.8788, 'longitude': 170.5028},
}

city_key = 'wellington'
forecast_days = 10
print(f'Notebook started at: {datetime.now():%Y-%m-%d %H:%M:%S}')

### API Request Preparation

This section constructs the parameters for the API request based on the selected city and desired forecast details.

-   `city`: Retrieves the latitude, longitude, and label for the chosen city from `CITY_COORDINATES`.
-   `params`: Defines the query parameters for the Open-Meteo API call, including `latitude`, `longitude`, requested `hourly` variables (temperature and relative humidity), `forecast_days`, and `timezone`.
-   `requests.Request('GET', BASE_URL, params=params).prepare()`: Creates a `PreparedRequest` object, which is useful for inspecting the request URL and headers before sending.

In [ ]:
city = CITY_COORDINATES[city_key]
params = {
    'latitude': city['latitude'],
    'longitude': city['longitude'],
    'hourly': 'temperature_2m,relative_humidity_2m',
    'forecast_days': forecast_days,
    'timezone': 'Pacific/Auckland',
}

prepared = requests.Request('GET', BASE_URL, params=params).prepare()
print('City:', city['label'])
print('Request URL:', prepared.url)
print(f'Equivalent curl: curl -i "{prepared.url}"')

### API Call and Response Handling

This cell executes the prepared API request, checks for a successful response, and parses the JSON content.

-   `requests.get()`: Sends an HTTP GET request to the Open-Meteo API.
-   `response.raise_for_status()`: Checks if the request was successful (status code 200). If not, it raises an HTTPError.
-   `content_type`: Verifies that the response is indeed JSON. An error is raised if the `Content-Type` header doesn't indicate JSON.
-   `response.json()`: Parses the JSON response body into a Python dictionary (`payload`).
-   The cell then prints the HTTP status code, content type, and the top-level keys found in the JSON payload to give an overview of the received data.

In [ ]:
response = requests.get(BASE_URL, params=params, headers={'Accept': 'application/json'}, timeout=15)
response.raise_for_status()

content_type = response.headers.get('Content-Type', '')
if 'application/json' not in content_type.lower():
    raise ValueError(f'Expected JSON but got: {content_type}')

payload = response.json()
print('Status code:', response.status_code)
print('Content-Type:', content_type)
print('Top-level keys:', sorted(payload.keys()))

### Extracting Data and Creating DataFrame

This section extracts specific hourly data from the JSON payload and organizes it into a Pandas DataFrame for easier manipulation and analysis.

-   `hourly = payload.get('hourly')`: Retrieves the 'hourly' dictionary, which contains arrays of time, temperature, and humidity.
-   **Validation**: It includes checks to ensure that 'hourly' data exists, that `time`, `temperature_2m`, and `relative_humidity_2m` are lists, and that all these lists have the same length. This prevents errors during DataFrame creation.
-   `pd.DataFrame(...)`: Creates a DataFrame with three columns:
    -   `time`: Converted to `datetime` objects using `pd.to_datetime()` for time-series analysis.
    -   `temperature_c`: Hourly temperature in Celsius.
    -   `humidity_percent`: Hourly relative humidity in percentage.
-   `df.head(8)`: Displays the first 8 rows of the newly created DataFrame to quickly inspect the data structure.

In [ ]:
hourly = payload.get('hourly')
if not isinstance(hourly, dict):
    raise ValueError("Missing 'hourly' object in payload")

times = hourly.get('time')
temperatures = hourly.get('temperature_2m')
humidities = hourly.get('relative_humidity_2m')

if not all(isinstance(v, list) for v in [times, temperatures, humidities]):
    raise ValueError('Expected list values for time, temperature_2m, relative_humidity_2m')
if not (len(times) == len(temperatures) == len(humidities)):
    raise ValueError('Hourly lists are not the same length')

df = pd.DataFrame({
    'time': pd.to_datetime(times),
    'temperature_c': temperatures,
    'humidity_percent': humidities,
})

df.head(8)



### Data Summary

This cell calculates and displays basic descriptive statistics for the temperature and humidity data in the DataFrame.

-   `summary` dictionary: Stores key metrics:
    -   `rows`: The total number of hourly data points.
    -   `temp_min`, `temp_max`, `temp_mean`: Minimum, maximum, and mean hourly temperatures.
    -   `humidity_mean`: Mean hourly humidity.
-   This provides a quick overview of the data's range and central tendencies.

In [ ]:
summary = {
    'rows': int(len(df)),
    'temp_min': float(df['temperature_c'].min()),
    'temp_max': float(df['temperature_c'].max()),
    'temp_mean': float(df['temperature_c'].mean()),
    'humidity_mean': float(df['humidity_percent'].mean()),
}
summary

### Data Visualisation

This section generates several plots to visualise the hourly temperature and humidity data, helping to understand trends and relationships.

#### Hourly Temperature Plot

-   `plt.figure(figsize=(11, 4.5))`: Creates a new figure with a specified size.
-   `plt.plot(df['time'], df['temperature_c'], ...)`: Plots the hourly temperature over time as a line graph.
-   The plot is titled with the city's label and includes labels for the X and Y axes, along with a grid for readability.

In [ ]:
plt.figure(figsize=(11, 4.5))
plt.plot(df['time'], df['temperature_c'], color='tab:red', linewidth=1.8)
plt.title(f"{city['label']} hourly temperature (C)")
plt.xlabel('Time')
plt.ylabel('Temperature (C)')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

#### Hourly Humidity Plot

-   Similar to the temperature plot, this cell generates a line plot showing the hourly relative humidity over time.
-   It uses a different colour (`tab:blue`) for distinction and includes appropriate titles and labels.

In [ ]:
plt.figure(figsize=(11, 4.5))
plt.plot(df['time'], df['humidity_percent'], color='tab:blue', linewidth=1.8)
plt.title(f"{city['label']} hourly humidity (%)")
plt.xlabel('Time')
plt.ylabel('Humidity (%)')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

#### Temperature vs. Humidity Scatter Plot

-   This plot helps visualize the relationship between hourly temperature and humidity.
-   `plt.scatter(df['temperature_c'], df['humidity_percent'], ...)`: Creates a scatter plot where each point represents an hourly reading of temperature and humidity.
-   The `alpha` parameter makes the points semi-transparent, which is useful for seeing data density in areas where many points overlap.

In [ ]:
plt.figure(figsize=(6.5, 5))
plt.scatter(df['temperature_c'], df['humidity_percent'], alpha=0.7, color='tab:green')
plt.title(f"{city['label']}: temperature vs humidity")
plt.xlabel('Temperature (C)')
plt.ylabel('Humidity (%)')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Daily Averages

This section calculates and visualises the daily average temperature and humidity, providing a higher-level overview of the forecast trends.

In [ ]:
df_daily_avg = df.set_index('time').resample('D').mean().reset_index()

display(df_daily_avg)

In [ ]:
fig_daily_temp = plt.figure(figsize=(11, 4.5))
plt.plot(df_daily_avg['time'], df_daily_avg['temperature_c'], color='tab:red', linewidth=2.5, marker='o')
plt.title(f"{city['label']} Daily Average Temperature (C)")
plt.xlabel('Date')
plt.ylabel('Average Temperature (C)')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
fig_daily_humidity = plt.figure(figsize=(11, 4.5))
plt.plot(df_daily_avg['time'], df_daily_avg['humidity_percent'], color='tab:blue', linewidth=2.5, marker='o')
plt.title(f"{city['label']} Daily Average Humidity (%)")
plt.xlabel('Date')
plt.ylabel('Average Humidity (%)')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Interactive Plots with Plotly Express

This section generates interactive line plots for hourly temperature and humidity using `plotly.express`. These plots allow for zooming, panning, and hovering to inspect individual data points, offering a more dynamic way to explore the time series data.

First, we ensure `plotly` is installed.

In [ ]:
try:
    import plotly.express as px
except ImportError:
    # Use !pip or %pip only in notebooks
    %pip install plotly
    import plotly.express as px

# Ensure 'df' and 'city' are defined before this point!
fig_interactive_temp = px.line(
    df,
    x='time',
    y='temperature_c',
    title=f"{city['label']} Hourly Temperature (C) - Interactive",
    labels={'time': 'Time', 'temperature_c': 'Temperature (C)'},
)

# Changed 'tab:red' to 'red' or a Hex code for compatibility
fig_interactive_temp.update_traces(line_color='#d62728')

fig_interactive_temp.show()

In [ ]:
fig_interactive_humidity = px.line(
    df,
    x='time',
    y='humidity_percent',
    title=f"{city['label']} Hourly Humidity (%) - Interactive",
    labels={'time': 'Time', 'humidity_percent': 'Humidity (%)'},
)

# Using the Hex code for 'tab:blue' for better compatibility
fig_interactive_humidity.update_traces(line_color='#1f77b4')

fig_interactive_humidity.show()

### Saving Outputs to Files

This final section saves the generated plots as PNG images and the raw JSON response as a text file to a specified output directory.

-   `output_dir = Path('notebook_outputs')`: Defines a `Path` object for the output directory.
-   `output_dir.mkdir(exist_ok=True)`: Creates the directory if it doesn't already exist.
-   **Saving Plots**: Each plot (temperature and humidity) is recreated within this section (using `plt.figure()`, `plt.plot()`, etc.) and then saved using `fig.savefig()`. `plt.close(fig)` is called after saving each figure to free up memory.
-   **Saving Raw JSON**: The original `response.text` (raw JSON string) is written to a file named after the city key (`<city_key>_raw.json`).
-   The absolute path of the output directory is printed to confirm where the files have been saved.

In [ ]:
output_dir = Path('notebook_outputs')
output_dir.mkdir(exist_ok=True)

fig1 = plt.figure(figsize=(11, 4.5))
plt.plot(df['time'], df['temperature_c'], color='tab:red', linewidth=1.8)
plt.title(f"{city['label']} hourly temperature (C)")
plt.xlabel('Time')
plt.ylabel('Temperature (C)')
plt.grid(alpha=0.25)
plt.tight_layout()
fig1.savefig(output_dir / f'{city_key}_temperature.png', dpi=140)
plt.close(fig1)

fig2 = plt.figure(figsize=(11, 4.5))
plt.plot(df['time'], df['humidity_percent'], color='tab:blue', linewidth=1.8)
plt.title(f"{city['label']} hourly humidity (%)")
plt.xlabel('Time')
plt.ylabel('Humidity (%)')
plt.grid(alpha=0.25)
plt.tight_layout()
fig2.savefig(output_dir / f'{city_key}_humidity.png', dpi=140)
plt.close(fig2)

(output_dir / f'{city_key}_raw.json').write_text(response.text, encoding='utf-8')
print('Saved files in:', output_dir.resolve())